In [1]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
path = str(Path.cwd().parent)
print(path)
sys.path.insert(1, path)

/home/joaquin/Documents/GitHub/skforecast


In [2]:
import numpy as np
import pandas as pd
import skforecast
print(skforecast.__version__)

0.21.0


In [3]:
lags_contiguous = np.arange(1, 25)          # [1..24]
lags_noncontig  = np.array([1, 2, 3, 6, 12, 24])

max_lag     = 24
window_size = max_lag   # no window features: window_size == max_lag

rng = np.random.default_rng(42)
y = rng.standard_normal(100_000)
y_strided = np.lib.stride_tricks.sliding_window_view(y, window_size)[:-1]
print(f"y_strided shape: {y_strided.shape}")

y_strided shape: (99976, 24)


In [4]:
# ── Correctness check ──────────────────────────────────────────────────────────
# Both methods must produce the same matrix for contiguous lags.
result_slice = y_strided[:, window_size - max_lag:][:, ::-1]  # view
result_fancy = y_strided[:, window_size - lags_contiguous]     # copy

assert np.array_equal(result_slice, result_fancy), "Results differ!"
print("Correctness check passed: both methods produce identical matrices.")
print(f"result_slice shares memory with y_strided : {np.shares_memory(result_slice, y_strided)}")
print(f"result_fancy shares memory with y_strided : {np.shares_memory(result_fancy, y_strided)}")

Correctness check passed: both methods produce identical matrices.
result_slice shares memory with y_strided : True
result_fancy shares memory with y_strided : False


In [5]:
%%timeit
# Contiguous lags 1-24 — slice + reverse (zero-copy view)
X_data = y_strided[:, window_size - max_lag:][:, ::-1]

764 ns ± 141 ns per loop (mean ± std. dev. of 7 runs, 1,000,000 loops each)


In [6]:
%%timeit
# Contiguous lags 1-24 — fancy index (forces a copy)
X_data = y_strided[:, window_size - lags_contiguous]

1.83 ms ± 93.4 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [7]:
%%timeit
# Non-contiguous lags [1,2,3,6,12,24] — fancy index (copy, fewer columns)
X_data = y_strided[:, window_size - lags_noncontig]

457 μs ± 55 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [8]:
from skforecast.recursive import ForecasterRecursive
from skforecast.direct import ForecasterDirect
from sklearn.linear_model import LinearRegression
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
y = pd.Series(rng.standard_normal(10_000), index=pd.date_range("2000-01-01", periods=10_000, freq="h"))
forecaster = ForecasterRecursive(
    estimator=LinearRegression(),
    lags=lags_contiguous,
)
forecaster_direct = ForecasterDirect(
    estimator=LinearRegression(),
    lags=lags_contiguous,
    steps=24
)

In [9]:
%%timeit

X_train, y_train = forecaster._create_lags(y)

181 μs ± 32.7 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [10]:
%%timeit

_ = forecaster_direct._create_lags(y)

55 μs ± 3.52 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


Impact in the prediction

In [11]:
forecaster = ForecasterRecursive(
    estimator=LinearRegression(),
    lags=lags_contiguous,
)


In [12]:
%%timeit
forecaster.fit(y=y, store_in_sample_residuals=True)

24.5 ms ± 368 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [13]:
%%timeit

_ = forecaster.predict(steps=500)

2.24 ms ± 56.6 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [14]:
%%timeit
_ = forecaster.predict_bootstrapping(steps=24, n_boot=500)

3.4 ms ± 102 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)
